# Pertemuan 13
## Information Flows and Technology

Membangun **Global Coffee Supply Chain Copilot** — AI yang bisa menjawab pertanyaan tentang data rantai pasok kopi global secara natural.

---
## 1. Dataset Rantai Pasok

In [2]:
import pandas as pd

df = pd.DataFrame({
    'tanggal':  pd.date_range('2024-01-01', periods=12, freq='ME'),
    'negara_asal': ['Indonesia', 'Vietnam', 'Brasil', 'Kolombia', 'Etiopia',
                    'Indonesia', 'Vietnam', 'Brasil', 'Kolombia', 'Etiopia',
                    'Indonesia', 'Vietnam'],
    'stok_ton':         [3200, 4100, 5200, 2800, 1900, 3000, 3900, 5000, 2600, 1800, 2900, 3700],
    'harga_usd_per_kg': [4.20, 3.95, 4.10, 4.55, 4.80, 4.35, 4.05, 4.20, 4.70, 4.95, 4.50, 4.15],
    'ekspor_ton':       [3800, 4600, 5800, 3200, 2200, 3500, 4400, 5600, 3000, 2100, 3400, 4200],
})

df

,tanggal,negara_asal,stok_ton,harga_usd_per_kg,ekspor_ton
0,2024-01-31,Indonesia,3200,4.20,3800
1,2024-02-29,Vietnam,4100,3.95,4600
2,2024-03-31,Brasil,5200,4.10,5800
3,2024-04-30,Kolombia,2800,4.55,3200
4,2024-05-31,Etiopia,1900,4.80,2200
5,2024-06-30,Indonesia,3000,4.35,3500
6,2024-07-31,Vietnam,3900,4.05,4400
7,2024-08-31,Brasil,5000,4.20,5600
8,2024-09-30,Kolombia,2600,4.70,3000
9,2024-10-31,Etiopia,1800,4.95,2100


---
## 2. Supply Chain Copilot

Konsep: **RAG (Retrieval Augmented Generation)** — data nyata disisipkan ke dalam prompt agar AI menjawab berdasarkan fakta.

In [ ]:
from groq import Groq
from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
client = Groq(api_key=os.environ["GROQ_API_KEY"])

# Ringkasan data sebagai konteks untuk AI
context = (
    "Data rantai pasok kopi global (Jan\u2013Des 2024):\n"
    + df.to_string(index=False)
    + f"\n\nStatistik:"
    f"\n- Harga tertinggi: USD {df['harga_usd_per_kg'].max():.2f}/kg ({df.loc[df['harga_usd_per_kg'].idxmax(), 'tanggal'].strftime('%b %Y')}, {df.loc[df['harga_usd_per_kg'].idxmax(), 'negara_asal']})"
    f"\n- Stok terendah: {df['stok_ton'].min():,} ton ({df.loc[df['stok_ton'].idxmin(), 'tanggal'].strftime('%b %Y')}, {df.loc[df['stok_ton'].idxmin(), 'negara_asal']})"
    f"\n- Negara dengan rata-rata ekspor terendah: {df.groupby('negara_asal')['ekspor_ton'].mean().idxmin()}"
)

def copilot(pertanyaan: str) -> str:
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "Kamu adalah Global Coffee Supply Chain Copilot. Gunakan data berikut untuk menjawab:\n" + context},
            {"role": "user",   "content": pertanyaan}
        ]
    )
    return response.choices[0].message.content

print("Supply Chain Copilot siap!")

Supply Chain Copilot siap!


---
## 3. Tanya Jawab dengan Copilot

In [4]:
print(copilot("Mengapa harga kopi global naik?"))

Harga kopi global dapat naik karena beberapa kemungkinan faktor, antara lain:

1. **Stok yang menurun**: Pada Oktober 2024, stok kopi dari Etiopia mencapai titik terendah sebesar 1.800 ton. Ketika stok dari salah satu negara penghasil utama menurun, pasokan global ikut tertekan dan harga cenderung naik.
2. **Variasi pasokan antarnegara**: Etiopia tercatat memiliki rata-rata volume ekspor terendah dibandingkan Brasil, Vietnam, Kolombia, dan Indonesia, sehingga gangguan pada negara ini berdampak lebih besar terhadap keseimbangan harga global.
3. **Fluktuasi musiman panen**: Negara-negara penghasil kopi memiliki musim panen yang berbeda-beda; ketika beberapa negara memasuki masa panen rendah secara bersamaan, pasokan global menurun dan harga naik.
4. **Biaya logistik dan pengiriman lintas benua**: Kenaikan biaya pengiriman kontainer dan bahan bakar kapal dapat meningkatkan biaya pokok kopi yang sampai ke pasar tujuan.
5. **Faktor eksternal**: Perubahan cuaca (misalnya kekeringan di Brasil

In [5]:
print(copilot("Kapan stok kopi mencapai titik terendah?"))

Stok kopi mencapai titik terendah pada Oktober 2024, yaitu sebesar 1.800 ton dari Etiopia.


In [6]:
print(copilot("Negara mana yang paling berisiko kekurangan pasokan ekspor?"))

Berdasarkan data yang ada, Etiopia paling berisiko mengalami kekurangan pasokan ekspor. Negara ini mencatat stok terendah sepanjang tahun (1.800 ton pada Oktober 2024) sekaligus rata-rata volume ekspor terendah dibandingkan Indonesia, Vietnam, Brasil, dan Kolombia. Kombinasi stok yang menipis dan volume ekspor yang relatif kecil membuat Etiopia paling rentan terhadap gangguan pasokan di rantai pasok kopi global.


---
## 4. Konsep RAG

| Tanpa RAG | Dengan RAG |
|---|---|
| AI menjawab dari pengetahuan umum | AI menjawab dari data spesifik yang diberikan |
| Tidak akurat untuk data internal | Akurat dan relevan |
| Tidak bisa diperbarui real-time | Bisa diperbarui dengan data baru |

RAG adalah fondasi dari banyak aplikasi enterprise AI saat ini.